In [1]:
import torch
import torch.nn as nn
import copy

class GptOssLayer(nn.Module):
    def __init__(self, config, alpha=1.702, limit=7.0):
        super().__init__()
        self.intermediate_size = config.intermediate_size
        self.hidden_size = config.hidden_size
        
        # Linear로 변경 (양자화 호환)
        self.gate_up_proj = nn.Linear(self.hidden_size, 2 * self.intermediate_size, bias=True)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=True)
        
        # 상수 설정 (외부에서 주입 가능하도록 변경)
        self.alpha = alpha
        self.limit = limit

    def forward(self, hidden_states):
        gate_up = self.gate_up_proj(hidden_states)
        gate = gate_up[..., ::2]
        up = gate_up[..., 1::2]
        
        # Limit 적용
        if self.limit is not None and self.limit > 0:
            gate = gate.clamp(min=None, max=self.limit)
            up = up.clamp(min=-self.limit, max=self.limit)
        
        glu = gate * torch.sigmoid(gate * self.alpha)
        return self.down_proj((up + 1) * glu)

class GptOssExperts(nn.Module):
    def __init__(self, config, alpha=1.702, limit=7.0):
        super().__init__()
        self.num_experts = config.num_local_experts
        self.hidden_size = config.hidden_size
        # 개별 Linear Expert 생성
        self.experts = nn.ModuleList([
            GptOssLayer(config, alpha, limit) for _ in range(self.num_experts)
        ])

    def forward(self, hidden_states: torch.Tensor, router_indices=None, routing_weights=None) -> torch.Tensor:
        batch_size = hidden_states.shape[0]
        hidden_states = hidden_states.reshape(-1, self.hidden_size)
        num_experts = routing_weights.shape[1]

        next_states = torch.zeros_like(hidden_states, dtype=hidden_states.dtype, device=hidden_states.device)
        
        with torch.no_grad():
            expert_mask = torch.nn.functional.one_hot(router_indices, num_classes=num_experts)
            expert_mask = expert_mask.permute(2, 1, 0)
            expert_hitted = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()

        for expert_idx in expert_hitted[:]:
            idx_int = expert_idx.item()
            expert_layer = self.experts[idx_int]

            with torch.no_grad():
                _, token_idx = torch.where(expert_mask[expert_idx[0]])
            
            current_state = hidden_states[token_idx]
            expert_output = expert_layer(current_state)
            
            # [0] 제거됨 (정상)
            weighted_output = expert_output * routing_weights[token_idx, idx_int].unsqueeze(-1)
            next_states.index_add_(0, token_idx, weighted_output.to(hidden_states.dtype))
        
        next_states = next_states.view(batch_size, -1, self.hidden_size)
        return next_states

class GptOssTopKRouter(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.top_k = config.num_experts_per_tok
        self.num_experts = config.num_local_experts
        self.hidden_dim = config.hidden_size
        self.gate = nn.Linear(self.hidden_dim, self.num_experts, bias=True)

    def forward(self, hidden_states):
        hidden_states = hidden_states.reshape(-1, self.hidden_dim)
        router_logits = self.gate(hidden_states)
        router_top_value, router_indices = torch.topk(router_logits, self.top_k, dim=-1)
        router_top_value = torch.nn.functional.softmax(router_top_value, dim=1, dtype=router_top_value.dtype)
        router_scores = torch.zeros_like(router_logits).scatter_(1, router_indices, router_top_value)
        return router_scores, router_indices

def replace_gptoss_modules(model):
    """
    원본 모델(GptOssForCausalLM)의 내부 모듈(Router, Experts)을 
    Standard 버전(nn.Linear 기반)으로 직접 교체합니다.
    """
    print("🔄 Standard Module 교체 작업 시작...")
    config = model.config
    
    # 모델의 모든 레이어 순회
    for i, layer in enumerate(model.model.layers):
        # -------------------------------------------------
        # A. 원본 모듈 가져오기 (참조)
        # -------------------------------------------------
        old_router = layer.mlp.router
        old_experts = layer.mlp.experts
        
        # [중요] 원본 모델의 alpha, limit 값 추출 (오차 방지)
        orig_alpha = getattr(old_experts, 'alpha', 1.702)
        orig_limit = getattr(old_experts, 'limit', 7.0)
        
        # -------------------------------------------------
        # B. 새 Router 생성 및 가중치 이식
        # -------------------------------------------------
        new_router = GptOssTopKRouter(config)
        new_router.to(old_router.weight.device).to(old_router.weight.dtype)
        
        # Router 가중치 복사 (nn.Parameter -> nn.Linear)
        # F.linear(x, w, b)에서 w는 (Out, In)이므로 그대로 복사
        new_router.gate.weight.data.copy_(old_router.weight.data)
        if old_router.bias is not None:
            new_router.gate.bias.data.copy_(old_router.bias.data)
            
        # -------------------------------------------------
        # C. 새 Experts 생성 및 가중치 이식 (핵심!)
        # -------------------------------------------------
        new_experts = GptOssExperts(config, alpha=orig_alpha, limit=orig_limit)
        new_experts.to(old_experts.gate_up_proj.device).to(old_experts.gate_up_proj.dtype)

        # 3D Tensor 슬라이싱 -> nn.Linear 이식
        # Old Shape: [Num_Experts, In, Out] (혹은 [N, H, 2I])
        # 확인: 원본 GateUp은 (N, Hidden, 2*Inter) / Down은 (N, Inter, Hidden)
        
        old_gate_up = old_experts.gate_up_proj.data # (N, H, 2I)
        old_gate_up_bias = old_experts.gate_up_proj_bias.data # (N, 2I)
        old_down = old_experts.down_proj.data # (N, I, H)
        old_down_bias = old_experts.down_proj_bias.data # (N, H)
        
        for idx in range(config.num_local_experts):
            target_expert = new_experts.experts[idx]
            
            # 1. Gate Up Proj
            # Old Slice: (Hidden, 2*Inter)
            # New Linear: (2*Inter, Hidden) -> Transpose(.t()) 필요
            w_gate = old_gate_up[idx] 
            b_gate = old_gate_up_bias[idx]
            target_expert.gate_up_proj.weight.data.copy_(w_gate.t())
            target_expert.gate_up_proj.bias.data.copy_(b_gate)
            
            # 2. Down Proj
            # Old Slice: (Inter, Hidden)
            # New Linear: (Hidden, Inter) -> Transpose(.t()) 필요
            w_down = old_down[idx]
            b_down = old_down_bias[idx]
            target_expert.down_proj.weight.data.copy_(w_down.t())
            target_expert.down_proj.bias.data.copy_(b_down)

        # -------------------------------------------------
        # D. 모듈 갈아끼우기 (Swap)
        # -------------------------------------------------
        layer.mlp.router = new_router
        layer.mlp.experts = new_experts
        
        if i == 0:
            print(f"   ℹ️ Layer 0 변환 완료 (Alpha: {orig_alpha}, Limit: {orig_limit})")

    print("✅ 모든 레이어의 Standard 모듈 교체가 완료되었습니다!")
    return model

# V2

In [1]:
import torch
import torch.nn as nn
import copy

class GptOssTopKRouter(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.top_k = config.num_experts_per_tok
        self.num_experts = config.num_local_experts
        self.hidden_dim = config.hidden_size
        self.gate = nn.Linear(self.hidden_dim, self.num_experts, bias=True)

    def forward(self, hidden_states):
        hidden_states = hidden_states.reshape(-1, self.hidden_dim)
        router_logits = self.gate(hidden_states)
        router_top_value, router_indices = torch.topk(router_logits, self.top_k, dim=-1)
        router_top_value = torch.nn.functional.softmax(router_top_value, dim=1, dtype=router_top_value.dtype)
        router_scores = torch.zeros_like(router_logits).scatter_(1, router_indices, router_top_value)
        return router_scores, router_indices
    
class GptOssExperts(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.intermediate_size = config.intermediate_size
        self.num_experts = config.num_local_experts
        self.hidden_size = config.hidden_size
        self.expert_dim = self.intermediate_size
        # self.gate_up_proj = nn.Parameter(torch.empty(self.num_experts, self.hidden_size, 2 * self.expert_dim))
        # self.gate_up_proj_bias = nn.Parameter(torch.empty(self.num_experts, 2 * self.expert_dim))
        self.gate_up_experts = nn.ModuleList([
            nn.Linear(self.hidden_size, 2 * self.intermediate_size, bias=True)
            for _ in range(self.num_experts)
        ])
        # self.down_proj = nn.Parameter(torch.empty((self.num_experts, self.expert_dim, self.hidden_size)))
        # self.down_proj_bias = nn.Parameter(torch.empty(self.num_experts, self.hidden_size))
        self.down_experts = nn.ModuleList([
            nn.Linear(self.intermediate_size, self.hidden_size, bias=True)
            for _ in range(self.num_experts)
        ])
        self.alpha = 1.702
        self.limit = 7.0

    def forward(self, hidden_states: torch.Tensor, router_indices=None, routing_weights=None) -> torch.Tensor:
        """
        When training is is more efficient to just loop over the experts and compute the output for each expert
        as otherwise the memory would explode.

        For inference we can sacrifice some memory and compute the output for all experts at once. By repeating the inputs.

        Args:
            hidden_states (torch.Tensor): (batch_size, seq_len, hidden_size)
            selected_experts (torch.Tensor): (batch_size * token_num, top_k)
            routing_weights (torch.Tensor): (batch_size * token_num, num_experts)
        Returns:
            torch.Tensor
        """
        batch_size = hidden_states.shape[0]
        hidden_states = hidden_states.reshape(-1, self.hidden_size)  # (num_tokens, hidden_size)
        num_experts = routing_weights.shape[1]
            
        # if hidden_states.device.type == "cpu" or self.training:
        if True:
            next_states = torch.zeros_like(hidden_states, dtype=hidden_states.dtype, device=hidden_states.device)
            with torch.no_grad():
                expert_mask = torch.nn.functional.one_hot(
                    router_indices, num_classes=num_experts + 1
                )  # masking is also a class
                expert_mask = expert_mask.permute(2, 1, 0)
                # we sum on the top_k and on the sequence length to get which experts
                # are hit this time around
                expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()
            for expert_idx in expert_hit[:]:
                # expert_idx only have 1 element, so we can use scale for fast indexing
                expert_idx = expert_idx[0]
                # skip masking index
                if expert_idx == num_experts:
                    continue
                with torch.no_grad():
                    _, token_idx = torch.where(expert_mask[expert_idx])
                current_state = hidden_states[token_idx]
                
                idx_int = expert_idx.item()
                gate_up_layer = self.gate_up_experts[idx_int]
                down_layer = self.down_experts[idx_int]
                gate_up = gate_up_layer(current_state)
                
                
                # gate_up = current_state @ self.gate_up_proj[expert_idx] + self.gate_up_proj_bias[expert_idx]
                gate, up = gate_up[..., ::2], gate_up[..., 1::2]
                gate = gate.clamp(min=None, max=self.limit)
                up = up.clamp(min=-self.limit, max=self.limit)
                glu = gate * torch.sigmoid(gate * self.alpha)
                gated_output = (up + 1) * glu
                out = down_layer(gated_output)
                # out = gated_output @ self.down_proj[expert_idx] + self.down_proj_bias[expert_idx]
                weighted_output = out * routing_weights[token_idx, expert_idx, None]
                next_states.index_add_(0, token_idx, weighted_output.to(hidden_states.dtype))
            next_states = next_states.view(batch_size, -1, self.hidden_size)
            
        else:
            hidden_states = hidden_states.repeat(num_experts, 1)
            hidden_states = hidden_states.view(num_experts, -1, self.hidden_size)
            gate_up = torch.bmm(hidden_states, self.gate_up_proj) + self.gate_up_proj_bias[..., None, :]
            gate, up = gate_up[..., ::2], gate_up[..., 1::2]
            gate = gate.clamp(min=None, max=self.limit)
            up = up.clamp(min=-self.limit, max=self.limit)
            glu = gate * torch.sigmoid(gate * self.alpha)
            next_states = torch.bmm(((up + 1) * glu), self.down_proj)
            next_states = next_states + self.down_proj_bias[..., None, :]
            next_states = next_states.view(num_experts, batch_size, -1, self.hidden_size)
            next_states = next_states * routing_weights.transpose(0, 1).view(num_experts, batch_size, -1)[..., None]
            next_states = next_states.sum(dim=0)
        return next_states

def replace_gptoss_modules_v2(model):
    """
    원본 모델(GptOssForCausalLM)의 내부 모듈(Router, Experts)을 
    Standard 버전(nn.Linear ModuleList 기반)으로 직접 교체합니다.
    """
    print("🔄 Standard Module 교체 작업 시작...")
    config = model.config
    
    # 모델의 모든 레이어 순회
    for i, layer in enumerate(model.model.layers):
        # -------------------------------------------------
        # A. 원본 모듈 가져오기 (참조)
        # -------------------------------------------------
        old_router = layer.mlp.router
        old_experts = layer.mlp.experts
        
        # [중요] 원본 모델의 alpha, limit 값 추출 (오차 방지)
        # 만약 원본에 값이 없으면 기본값(1.702, 7.0) 사용
        orig_alpha = getattr(old_experts, 'alpha', 1.702)
        orig_limit = getattr(old_experts, 'limit', 7.0)
        
        # -------------------------------------------------
        # B. 새 Router 생성 및 가중치 이식
        # -------------------------------------------------
        new_router = GptOssTopKRouter(config)
        new_router.to(old_router.weight.device).to(old_router.weight.dtype)
        
        # Router 가중치 복사 (nn.Parameter -> nn.Linear)
        new_router.gate.weight.data.copy_(old_router.weight.data)
        if old_router.bias is not None:
            new_router.gate.bias.data.copy_(old_router.bias.data)
            
        # -------------------------------------------------
        # C. 새 Experts 생성 및 가중치 이식 (핵심!)
        # -------------------------------------------------
        # 1. 객체 생성 (일단 기본값으로 생성됨)
        new_experts = GptOssExperts(config)
        
        # 2. 상수 동기화 (원본 모델 값으로 덮어쓰기)
        new_experts.alpha = orig_alpha
        new_experts.limit = orig_limit
        
        # Device/Dtype 이동
        new_experts.to(old_experts.gate_up_proj.device).to(old_experts.gate_up_proj.dtype)

        # 3. 원본 3D 텐서 데이터 가져오기
        # Old Shape: [Num_Experts, Hidden, 2*Intermediate]
        old_gate_up = old_experts.gate_up_proj.data 
        old_gate_up_bias = old_experts.gate_up_proj_bias.data 
        
        # Old Shape: [Num_Experts, Intermediate, Hidden]
        old_down = old_experts.down_proj.data 
        old_down_bias = old_experts.down_proj_bias.data 
        
        # 4. 각 Expert별로 분리해서 nn.Linear에 이식
        for idx in range(config.num_local_experts):
            # (1) Gate Up Layer 이식
            # Target: new_experts.gate_up_experts[idx]
            # Old Slice: (Hidden, 2*Inter) -> nn.Linear: (2*Inter, Hidden) 이므로 Transpose 필요
            w_gate = old_gate_up[idx] 
            b_gate = old_gate_up_bias[idx]
            
            new_experts.gate_up_experts[idx].weight.data.copy_(w_gate.t())
            new_experts.gate_up_experts[idx].bias.data.copy_(b_gate)
            
            # (2) Down Layer 이식
            # Target: new_experts.down_experts[idx]
            # Old Slice: (Inter, Hidden) -> nn.Linear: (Hidden, Inter) 이므로 Transpose 필요
            w_down = old_down[idx]
            b_down = old_down_bias[idx]
            
            new_experts.down_experts[idx].weight.data.copy_(w_down.t())
            new_experts.down_experts[idx].bias.data.copy_(b_down)

        # -------------------------------------------------
        # D. 모듈 갈아끼우기 (Swap)
        # -------------------------------------------------
        layer.mlp.router = new_router
        layer.mlp.experts = new_experts
        
        if i == 0:
            print(f"   ℹ️ Layer 0 변환 완료 (Alpha: {orig_alpha}, Limit: {orig_limit})")

    print("✅ 모든 레이어의 Standard 모듈 교체가 완료되었습니다!")
    return model

In [2]:
import torch
import torch.nn as nn
import copy
class GptOssExperts(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.intermediate_size = config.intermediate_size
        self.num_experts = config.num_local_experts
        self.hidden_size = config.hidden_size
        self.expert_dim = self.intermediate_size
        self.gate_up_proj = nn.Parameter(torch.empty(self.num_experts, self.hidden_size, 2 * self.expert_dim))
        self.gate_up_proj_bias = nn.Parameter(torch.empty(self.num_experts, 2 * self.expert_dim))
        self.down_proj = nn.Parameter(torch.empty((self.num_experts, self.expert_dim, self.hidden_size)))
        self.down_proj_bias = nn.Parameter(torch.empty(self.num_experts, self.hidden_size))
        self.alpha = 1.702
        self.limit = 7.0

    def forward(self, hidden_states: torch.Tensor, router_indices=None, routing_weights=None) -> torch.Tensor:
        """
        When training it is more efficient to just loop over the experts and compute the output for each expert
        as otherwise the memory would explode.

        For inference we can sacrifice some memory and compute the output for all experts at once. By repeating the inputs.

        Args:
            hidden_states (torch.Tensor): (batch_size, seq_len, hidden_size)
            selected_experts (torch.Tensor): (batch_size * token_num, top_k)
            routing_weights (torch.Tensor): (batch_size * token_num, num_experts)
        Returns:
            torch.Tensor
        """
        batch_size = hidden_states.shape[0]
        hidden_states = hidden_states.reshape(-1, self.hidden_size)  # (num_tokens, hidden_size)
        num_experts = routing_weights.shape[1]
        # if hidden_states.device.type == "cpu" or self.training:
        if True:
            next_states = torch.zeros_like(hidden_states, dtype=hidden_states.dtype, device=hidden_states.device)
            with torch.no_grad():
                expert_mask = torch.nn.functional.one_hot(
                    router_indices, num_classes=num_experts + 1
                )  # masking is also a class
                expert_mask = expert_mask.permute(2, 1, 0)
                # we sum on the top_k and on the sequence length to get which experts
                # are hit this time around
                expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()
            for expert_idx in expert_hit[:]:
                # expert_idx only have 1 element, so we can use scale for fast indexing
                expert_idx = expert_idx[0]
                # skip masking index
                if expert_idx == num_experts:
                    continue
                with torch.no_grad():
                    _, token_idx = torch.where(expert_mask[expert_idx])
                current_state = hidden_states[token_idx]
                gate_up = current_state @ self.gate_up_proj[expert_idx] + self.gate_up_proj_bias[expert_idx]
                gate, up = gate_up[..., ::2], gate_up[..., 1::2]
                gate = gate.clamp(min=None, max=self.limit)
                up = up.clamp(min=-self.limit, max=self.limit)
                glu = gate * torch.sigmoid(gate * self.alpha)
                gated_output = (up + 1) * glu
                out = gated_output @ self.down_proj[expert_idx] + self.down_proj_bias[expert_idx]
                weighted_output = out * routing_weights[token_idx, expert_idx, None]
                next_states.index_add_(0, token_idx, weighted_output.to(hidden_states.dtype))
            next_states = next_states.view(batch_size, -1, self.hidden_size)
        else:
            hidden_states = hidden_states.repeat(num_experts, 1)
            hidden_states = hidden_states.view(num_experts, -1, self.hidden_size)
            gate_up = torch.bmm(hidden_states, self.gate_up_proj) + self.gate_up_proj_bias[..., None, :]
            gate, up = gate_up[..., ::2], gate_up[..., 1::2]
            gate = gate.clamp(min=None, max=self.limit)
            up = up.clamp(min=-self.limit, max=self.limit)
            glu = gate * torch.sigmoid(gate * self.alpha)
            next_states = torch.bmm(((up + 1) * glu), self.down_proj)
            next_states = next_states + self.down_proj_bias[..., None, :]
            next_states = next_states.view(num_experts, batch_size, -1, self.hidden_size)
            next_states = next_states * routing_weights.transpose(0, 1).view(num_experts, batch_size, -1)[..., None]
            next_states = next_states.sum(dim=0)
        return next_states

def replace_gptoss_modules_v3(model):
    """
    GptOssExperts를 V3 버전(nn.Linear + Robust Logic)으로 교체합니다.
    """
    print("🔄 Standard Module V3 교체 작업 시작...")
    config = model.config
    
    for i, layer in enumerate(model.model.layers):

        old_experts = layer.mlp.experts # 3D Tensor 방식
        
        orig_alpha = getattr(old_experts, 'alpha', 1.702)
        orig_limit = getattr(old_experts, 'limit', 7.0)
        
        new_experts = GptOssExperts(config)
        new_experts.to(old_experts.gate_up_proj.device).to(old_experts.gate_up_proj.dtype)

        # 원본 3D 텐서 데이터 가져오기
        old_gate_up = old_experts.gate_up_proj.data # (Num, Hidden, 2*Inter)
        old_gate_up_bias = old_experts.gate_up_proj_bias.data 
        old_down = old_experts.down_proj.data # (Num, Inter, Hidden)
        old_down_bias = old_experts.down_proj_bias.data
         
        new_experts.gate_up_proj.data.copy_(old_gate_up)
        new_experts.gate_up_proj_bias.data.copy_(old_gate_up_bias)
        new_experts.down_proj.data.copy_(old_down)
        new_experts.down_proj_bias.data.copy_(old_down_bias)
        
        layer.mlp.experts = new_experts
        
        if i == 0:
            print(f"   ℹ️ Layer 0 변환 완료 (Alpha: {orig_alpha}, Limit: {orig_limit})")

    print("✅ 모든 레이어의 GptOssExperts V3 교체가 완료되었습니다!")
    return model

In [1]:
def transfer_weights(orig_model, std_model):
    print("🔄 가중치 변환 및 이식 시작...")
    
    orig_state_dict = orig_model.state_dict()
    std_state_dict = std_model.state_dict()
    
    new_state_dict = {}
    num_experts = orig_model.config.num_local_experts
    # target_dtype = orig_model.model.embed_tokens.weight.dtype

    for key, value in orig_state_dict.items():
        if "router.weight" in key:
            new_key = key.replace("router.weight", "router.gate.weight")
            new_state_dict[new_key] = value
            
        elif "router.bias" in key:
            new_key = key.replace("router.bias", "router.gate.bias")
            new_state_dict[new_key] = value
            
        elif "experts.gate_up_proj" in key:
            # key: ...mlp.experts.gate_up_proj (3D Tensor)
            # value shape: [num_experts, hidden, 2*inter]
            
            if "gate_up_proj_bias" in key: # Bias 처리
                base_key = key.replace("experts.gate_up_proj_bias", "experts.experts") # ...mlp.experts.experts
                for i in range(num_experts):
                    # target: ...mlp.experts.experts.0.gate_up_proj.bias
                    target_key = f"{base_key}.{i}.gate_up_proj.bias"
                    new_state_dict[target_key] = value[i]
            else: # Weight 처리
                base_key = key.replace("experts.gate_up_proj", "experts.experts") # ...mlp.experts.experts
                for i in range(num_experts):
                    # target: ...mlp.experts.experts.0.gate_up_proj.weight
                    target_key = f"{base_key}.{i}.gate_up_proj.weight"
                    new_state_dict[target_key] = value[i].t()

        elif "experts.down_proj" in key:
            # key: ...mlp.experts.down_proj (3D Tensor)
            # value shape: [num_experts, inter, hidden]
            
            if "down_proj_bias" in key: # Bias 처리
                # value shape: [num_experts, hidden]
                base_key = key.replace("experts.down_proj_bias", "experts.experts")
                for i in range(num_experts):
                    target_key = f"{base_key}.{i}.down_proj.bias"
                    new_state_dict[target_key] = value[i]
            else: # Weight 처리
                # value shape: [num_experts, inter, hidden]
                base_key = key.replace("experts.down_proj", "experts.experts")
                for i in range(num_experts):
                    target_key = f"{base_key}.{i}.down_proj.weight"
                    
                    new_state_dict[target_key] = value[i].t()
        else:
            new_state_dict[key] = value

    # strict=True로 해서 키가 하나라도 안 맞으면 에러가 나게 검증
    std_model.load_state_dict(new_state_dict, strict=True)
    print("✅ 가중치 이식 완료!")


# Model load

In [2]:
import torch
import copy
from transformers import AutoTokenizer, AutoConfig

from transformers import GptOssForCausalLM 
# from gptoss_original import GptOssForCausalLM

from gptoss_standard_moe_v1 import GptOssForCausalLM as GptOssForCausalLM_standard

model_id = 'openai/gpt-oss-20b' # 실제 존재하는 모델 ID여야 함 (예시)
cache_directory = '/home/jgryu/.cache/huggingface/hub'
config = AutoConfig.from_pretrained(model_id, cache_dir=cache_directory)

print("1. 원본 모델 로딩 중... (Eager Mode)")
orig_model = GptOssForCausalLM.from_pretrained(
    model_id, 
    cache_dir=cache_directory, 
    torch_dtype=torch.bfloat16,
    # torch_dtype=torch.float32,
    device_map="cpu",
    # attn_implementation='sdpa',
)

print("2. Standard 모델 초기화 중...")
# std_model = GptOssForCausalLM_standard(config) # 리팩토링 모델
# std_model = GptOssForCausalLM_standard.from_pretrained(
#     model_id, 
#     config=config,
#     cache_dir=cache_directory, 
#     torch_dtype=torch.bfloat16,
#     # torch_dtype=torch.float32,
#     device_map="cpu"
# )
# std_model = GptOssForCausalLM.from_pretrained(
#     model_id, 
#     cache_dir=cache_directory, 
#     torch_dtype=torch.bfloat16,
#     # torch_dtype=torch.float32,
#     device_map="cpu"
# )
# std_model.to(dtype=torch.bfloat16)  ## 이걸 하면 attn 부터 깨짐. 왜지?

# 이걸 하면 깨짐. 제미나이 개같은거
# std_model.to(orig_model.dtype)
# std_model.to(orig_model.device)

# 3. 가중치 복사
print("3. State Dict 로딩...")
# transfer_weights(orig_model, std_model) # 리팩토링 모델일 경우 사용
# std_model = replace_gptoss_modules(std_model)
# std_model = replace_gptoss_modules_v2(std_model)
# std_model = replace_gptoss_modules_v3(std_model)
# std_model.load_state_dict(orig_model.state_dict(), strict=True) 

# 4. [핵심] 누락된 Buffer(RoPE inv_freq 등) 강제 동기화
# state_dict에는 parameter만 있고 buffer가 없을 수 있으므로 강제로 덮어씁니다.
# print("4. Buffer 동기화 (RoPE 등)...")
# for (name_orig, buffer_orig), (name_std, buffer_std) in zip(orig_model.named_buffers(), std_model.named_buffers()):
#     if name_orig == name_std:
# #         buffer_std.data.copy_(buffer_orig.data)

# print("\n🧪 동작 검증 시작...")
# device_std = 'cuda:4'
# device_orig = 'cuda:5'
# device_comp = 'cuda:6'
# std_model.to(device_std)
# orig_model.to(device_orig)

# tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_directory)

`torch_dtype` is deprecated! Use `dtype` instead!


1. 원본 모델 로딩 중... (Eager Mode)


MXFP4 quantization requires Triton and kernels installed: CUDA requires Triton >= 3.4.0, XPU requires Triton >= 3.5.0, we will default to dequantizing the model to bf16
Loading checkpoint shards: 100%|██████████| 3/3 [00:16<00:00,  5.47s/it]


2. Standard 모델 초기화 중...
3. State Dict 로딩...


In [1]:
from transformers import AutoTokenizer

model_id = 'openai/gpt-oss-20b' # 실제 존재하는 모델 ID여야 함 (예시)
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(tokenizer.chat_template)

/opt/conda/envs/gptoss/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{#-
  In addition to the normal inputs of `messages` and `tools`, this template also accepts the
  following kwargs:
  - "builtin_tools": A list, can contain "browser" and/or "python".
  - "model_identity": A string that optionally describes the model identity.
  - "reasoning_effort": A string that describes the reasoning effort, defaults to "medium".
 #}

{#- Tool Definition Rendering ============================================== #}
{%- macro render_typescript_type(param_spec, required_params, is_nullable=false) -%}
    {%- if param_spec.type == "array" -%}
        {%- if param_spec['items'] -%}
            {%- if param_spec['items']['type'] == "string" -%}
                {{- "string[]" }}
            {%- elif param_spec['items']['type'] == "number" -%}
                {{- "number[]" }}
            {%- elif param_spec['items']['type'] == "integer" -%}
                {{- "number[]" }}
            {%- elif param_spec['items']['type'] == "boolean" -%}
                {{- "boolean[]" }

In [ ]:
print(f"\n🔍 [Original Model] Tensor Shape & Info Inspection")
print("="*80)
print(f"{'Name':<60} | {'Shape':<25} | {'Dtype':<10} | {'Device':<10}")
print("-" * 80)

target_keyword = "layers.0.mlp" # 보고 싶은 모듈 키워드

print(f"\n🔍 Filtering for keyword: '{target_keyword}'")
for name, param in orig_model.named_parameters():
    if target_keyword in name:
        print(f"{name:<60} | {str(list(param.shape)):<25}")


🔍 [Original Model] Tensor Shape & Info Inspection
Name                                                         | Shape                     | Dtype      | Device    
--------------------------------------------------------------------------------

🔍 Filtering for keyword: 'layers.0.mlp'
model.layers.0.mlp.router.weight                             | [32, 2880]               
model.layers.0.mlp.router.bias                               | [32]                     
model.layers.0.mlp.experts.gate_up_proj                      | [32, 2880, 5760]         
model.layers.0.mlp.experts.gate_up_proj_bias                 | [32, 5760]               
model.layers.0.mlp.experts.down_proj                         | [32, 2880, 2880]         
model.layers.0.mlp.experts.down_proj_bias                    | [32, 2880]               


: 

In [ ]:
# def print_dtype(orig_model, std_model):
    
#     orig_state_dict = orig_model.state_dict()
#     std_state_dict = std_model.state_dict()
    
#     new_state_dict = {}
#     num_experts = orig_model.config.num_local_experts
#     # target_dtype = orig_model.model.embed_tokens.weight.dtype
#     print("\n🔎 [Debug] Original Model vs Standard Model Dtypes:")
    
#     # 1. Original Model Dtype 확인 (샘플 몇 개만)
#     print(f"   [Original Model] Total Keys: {len(orig_state_dict)}")
#     for i, (k, v) in enumerate(orig_state_dict.items()):
#         # if i < 5 or "experts" in k: # 처음 5개와 experts 관련 키만 출력
#         if i < 30:
#             print(f"    - {k}: {v.dtype} {v.shape}")
#             if i == 4: print("    ... (skip) ...")
    
#     print("-" * 50)

#     # 2. Standard Model Dtype 확인 (이식 전 상태)
#     print(f"   [Standard Model (Before Load)] Total Keys: {len(std_state_dict)}")
#     for i, (k, v) in enumerate(std_state_dict.items()):
#         # if i < 5 or "experts" in k:
#         if i < 30:
#             print(f"    - {k}: {v.dtype} {v.shape}")
#             if i == 4: print("    ... (skip) ...")
    
#     print("-" * 50 + "\n")

# print_dtype(orig_model, std_model)

In [3]:
import torch


orig_model.train()
# orig_model.eval()
std_model.eval()

orig_dump = {}
std_dump = {}

def hook_router(storage, name):
    def hook(model, input, output):
        # Router 출력: (scores, indices)
        storage[f"{name}_scores"] = output[0].detach().cpu().float()
        storage[f"{name}_indices"] = output[1].detach().cpu()
    return hook

def hook_experts(storage, name):
    def hook(model, input, output):
        # Experts 출력: Tensor
        storage[f"{name}_out"] = output.detach().cpu().float()
    return hook

def hook_mlp_input(storage, name):
    def hook(model, input, output):
        # MLP 입력: (hidden_states,) 튜플
        storage[f"{name}_in"] = input[0].detach().cpu().float()
    return hook

# ==========================================
# 2. Hook 등록 (Layer 0 집중 분석)
# ==========================================
# 모델 구조에 따라 접근 경로가 다를 수 있으니 확인 필요
# 보통: model.layers[0].mlp

# [Original Model Hook]
layer_orig = orig_model.model.layers[0].mlp
# layer_orig.training = True
layer_orig.register_forward_hook(hook_mlp_input(orig_dump, "MLP"))
layer_orig.router.register_forward_hook(hook_router(orig_dump, "Router"))
layer_orig.experts.register_forward_hook(hook_experts(orig_dump, "Experts"))
# layer_orig.experts.training = False

# [Standard Model Hook]
layer_std = std_model.model.layers[0].mlp
layer_std.register_forward_hook(hook_mlp_input(std_dump, "MLP"))
layer_std.router.register_forward_hook(hook_router(std_dump, "Router"))
layer_std.experts.register_forward_hook(hook_experts(std_dump, "Experts"))
# layer_orig.experts.training = False

print("🪝 Hook 설치 완료. 내부 값을 캡처할 준비가 되었습니다.")

# ==========================================
# 3. Forward 실행
# ==========================================
device_orig = next(orig_model.parameters()).device
device_std = next(std_model.parameters()).device

# 더미 입력
input_text = "Hello, verifying MLP internals."
inputs = tokenizer(input_text, return_tensors="pt")

print("\n🚀 Forward 실행 중...")
with torch.no_grad():
    _ = orig_model(**inputs.to(device_orig))
    _ = std_model(**inputs.to(device_std))

# ==========================================
# 4. 단계별 정밀 비교
# ==========================================
print("\n🕵️‍♀️ [MLP 내부 해부 결과]")

# (1) MLP 입력값 비교 (여기서 다르면 이전 레이어 문제)
diff_in = (orig_dump["MLP_in"] - std_dump["MLP_in"]).abs()
print(f"\n1️⃣ MLP 입력값 (Input Hidden States)")
print(f"   - Max Diff: {diff_in.max().item():.8f}")
if diff_in.max().item() > 1e-4:
    print("   🚨 [WARNING] 입력부터 이미 다릅니다! 이전 레이어(Attention/Norm)를 보세요.")
else:
    print("   ✅ 입력값 정상 (동일함)")

# (2) Router Indices 비교 (정수형, 무조건 똑같아야 함)
indices_orig = orig_dump["Router_indices"].long()
indices_std = std_dump["Router_indices"].long()
# 같은지 비교 (True/False)
match = (indices_orig == indices_std).all().item()

print(f"\n2️⃣ Router Indices (선택된 전문가 번호)")
print(f"   - 원본(일부): {indices_orig[0, :5].tolist()} ...")
print(f"   - 표준(일부): {indices_std[0, :5].tolist()} ...")
if match:
    print("   ✅ Indices 완전 일치 (라우팅 로직 정상)")
else:
    print("   ❌ [FAIL] 서로 다른 전문가를 선택했습니다!")
    print("      -> Router의 가중치(gate.weight)가 잘못 이식되었거나, 입력값이 달라서 발생했습니다.")

# (3) Router Scores 비교 (확률값)
diff_scores = (orig_dump["Router_scores"] - std_dump["Router_scores"]).abs()
print(f"\n3️⃣ Router Scores (Top-K 확률값)")
print(f"   - Max Diff: {diff_scores.max().item():.8f}")
if diff_scores.max().item() > 1e-4:
    print("   ❌ [FAIL] 확률값이 다릅니다. Router의 Linear 연산이나 Softmax 순서를 확인하세요.")
else:
    print("   ✅ Scores 정상 (가중치 연산 정상)")

# (4) Experts Output 비교 (최종 결과)
diff_out = (orig_dump["Experts_out"] - std_dump["Experts_out"]).abs()
print(f"\n4️⃣ Experts Output (최종 결과)")
print(f"   - Max Diff: {diff_out.max().item():.8f}")
print(f"   - Max Mean: {diff_out.mean().item():.8f}")

if diff_out.max().item() > 1e-3:
    if match and diff_scores.max().item() < 1e-4:
        print("   🚨 [결론] 입력/라우팅/확률은 다 맞는데 결과만 틀립니다.")
        print("      -> 범인은 'GptOssLayer' 내부 로직입니다.")
        print("      -> 체크포인트: 1) 가중치 Transpose 여부, 2) Activation(+1), 3) Slicing 순서")
    else:
        print("   🚨 [결론] 앞 단계(Indices/Scores)가 틀려서 결과도 틀린 것입니다.")
else:
    print("   🎉 [SUCCESS] MLP 내부 동작이 완벽하게 일치합니다.")

# # 메모리 정리
# orig_dump.clear()
# std_dump.clear()

🪝 Hook 설치 완료. 내부 값을 캡처할 준비가 되었습니다.

🚀 Forward 실행 중...

🕵️‍♀️ [MLP 내부 해부 결과]

1️⃣ MLP 입력값 (Input Hidden States)
   - Max Diff: 0.00000000
   ✅ 입력값 정상 (동일함)

2️⃣ Router Indices (선택된 전문가 번호)
   - 원본(일부): [5, 18, 9, 16] ...
   - 표준(일부): [5, 18, 9, 16] ...
   ✅ Indices 완전 일치 (라우팅 로직 정상)

3️⃣ Router Scores (Top-K 확률값)
   - Max Diff: 0.00000000
   ✅ Scores 정상 (가중치 연산 정상)

4️⃣ Experts Output (최종 결과)
   - Max Diff: 0.25000000
   - Max Mean: 0.00149724
   🚨 [결론] 입력/라우팅/확률은 다 맞는데 결과만 틀립니다.
      -> 범인은 'GptOssLayer' 내부 로직입니다.
      -> 체크포인트: 1) 가중치 Transpose 여부, 2) Activation(+1), 3) Slicing 순서


In [ ]:
import torch

def verify_weight_transfer_details(orig_model, std_model):
    print("🕵️ 가중치 이식 정밀 검증 시작...\n")
    
    # 1. 모델 준비 (CPU로 이동하여 비교, GPU 메모리 절약)
    device = torch.device("cpu")
    orig_model = orig_model.to(device)
    std_model = std_model.to(device)
    
    all_passed = True
    
    # GptOss Config에서 레이어 수와 전문가 수 가져오기
    num_layers = orig_model.config.num_hidden_layers
    num_experts = orig_model.config.num_local_experts
    
    # --- [검증 1] 일반 레이어 (Embeddings, Norm) ---
    print("🔹 1. 일반 레이어 검증 (Embeddings, Norm)")
    
    # Embed Tokens
    w_orig = orig_model.model.embed_tokens.weight
    w_std = std_model.model.embed_tokens.weight
    if torch.equal(w_orig, w_std):
        print(f"   ✅ Embed Tokens: 일치")
    else:
        print(f"   ❌ Embed Tokens: 불일치! Max Diff: {(w_orig - w_std).abs().max().item()}")
        all_passed = False

    # Final Norm
    w_orig = orig_model.model.norm.weight
    w_std = std_model.model.norm.weight
    if torch.equal(w_orig, w_std):
        print(f"   ✅ Final Norm: 일치")
    else:
        print(f"   ❌ Final Norm: 불일치!")
        all_passed = False
        
    print("-" * 30)

    # --- [검증 2] Transformer Layers (Router & Experts) ---
    print(f"🔹 2. Layer별 Router 및 Expert 검증 (총 {num_layers}개 Layer)")
    
    for i in range(num_layers):
        layer_orig = orig_model.model.layers[i]
        layer_std = std_model.model.layers[i]
        
        # 2-1. Router 검증
        # 원본: layer.mlp.router.weight / bias
        # 표준: layer.mlp.router.gate.weight / bias
        
        # Weight
        r_w_orig = layer_orig.mlp.router.weight
        r_w_std = layer_std.mlp.router.gate.weight
        if not torch.equal(r_w_orig, r_w_std):
            print(f"   ❌ [Layer {i}] Router Weight 불일치")
            all_passed = False
            
        # Bias
        r_b_orig = layer_orig.mlp.router.bias
        r_b_std = layer_std.mlp.router.gate.bias
        if not torch.equal(r_b_orig, r_b_std):
            print(f"   ❌ [Layer {i}] Router Bias 불일치")
            all_passed = False

        # 2-2. Experts 검증 (가장 중요: Transpose 확인)
        # 원본: layer.mlp.experts.gate_up_proj [Num_Experts, Hidden, 2*Inter]
        # 표준: layer.mlp.experts.experts[k].gate_up_proj.weight [2*Inter, Hidden] (Transposed!)
        
        orig_gate_up = layer_orig.mlp.experts.gate_up_proj
        orig_gate_up_bias = layer_orig.mlp.experts.gate_up_proj_bias
        
        orig_down = layer_orig.mlp.experts.down_proj
        orig_down_bias = layer_orig.mlp.experts.down_proj_bias
        
        for k in range(num_experts):
            std_expert = layer_std.mlp.experts.experts[k]
            
            # (A) Gate Up Proj Weight (Transpose 체크)
            w_src = orig_gate_up[k]      # [Hidden, 2*Inter]
            w_tgt = std_expert.gate_up_proj.weight # [2*Inter, Hidden]
            
            # 중요: w_src를 전치(.t()) 해야 w_tgt와 같아야 함
            if not torch.equal(w_src.t(), w_tgt):
                print(f"   ❌ [Layer {i} - Expert {k}] GateUp Weight Transpose 실패!")
                print(f"      Orig: {w_src.shape}, Std: {w_tgt.shape}")
                all_passed = False

            # (B) Gate Up Proj Bias
            b_src = orig_gate_up_bias[k]
            b_tgt = std_expert.gate_up_proj.bias
            if not torch.equal(b_src, b_tgt):
                print(f"   ❌ [Layer {i} - Expert {k}] GateUp Bias 불일치")
                all_passed = False
                
            # (C) Down Proj Weight (Transpose 체크)
            w_src = orig_down[k]        # [Inter, Hidden]
            w_tgt = std_expert.down_proj.weight # [Hidden, Inter]
            
            if not torch.equal(w_src.t(), w_tgt):
                print(f"   ❌ [Layer {i} - Expert {k}] DownProj Weight Transpose 실패!")
                all_passed = False

            # (D) Down Proj Bias
            b_src = orig_down_bias[k]
            b_tgt = std_expert.down_proj.bias
            if not torch.equal(b_src, b_tgt):
                print(f"   ❌ [Layer {i} - Expert {k}] DownProj Bias 불일치")
                all_passed = False
        
        # 첫 번째 레이어만 상세 출력하고 나머지는 생략 (로그 폭탄 방지)
        if i == 0:
            print(f"   ✅ [Layer 0] Router & All {num_experts} Experts 검증 통과 (Sample)")
            
    print("-" * 30)
    
    if all_passed:
        print("🎉 [SUCCESS] 모든 가중치가 완벽하게 이식되었습니다!")
        print("   - Router: 이름 변경 완료")
        print("   - Experts: 3D Tensor 분할 및 Transpose 완료")
        print("   - Bias: 정상 복사 완료")
    else:
        print("💥 [FAIL] 가중치 이식 과정에 오류가 있습니다.")

# 실행
verify_weight_transfer_details(orig_model, std_model)

In [4]:
orig_model.train()
# orig_model.eval()
std_model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_directory)

input_text = "Hello, I want to verify the model structure."
inputs = tokenizer(input_text, return_tensors="pt").to(orig_model.device)

with torch.no_grad():
    print("   - 원본 모델 Forward...")
    orig_outputs = orig_model(**inputs.to(device_orig))
    orig_logits = orig_outputs.logits
    
    print("   - Standard 모델 Forward...")
    std_outputs = std_model(**inputs.to(device_std))
    std_logits = std_outputs.logits

    # ------------------ [추가된 부분] ------------------
    # 로짓(Logits)에서 가장 확률이 높은 다음 토큰 ID 추출 (마지막 시퀀스 기준)
    orig_next_token_id = torch.argmax(orig_logits[:, -1, :], dim=-1)
    std_next_token_id = torch.argmax(std_logits[:, -1, :], dim=-1)

    # ID를 텍스트로 변환
    orig_pred_text = tokenizer.decode(orig_next_token_id)
    std_pred_text = tokenizer.decode(std_next_token_id)

    print(f"\n💬 모델 출력 비교 (Next Token):")
    print(f"   - 원본 모델 예측: '{orig_pred_text}'")
    print(f"   - Standard 예측 : '{std_pred_text}'")
    # ---------------------------------------------------

diff = (orig_logits.to(device_comp) - std_logits.to(device_comp)).abs()
max_diff = diff.max().item()
mean_diff = diff.mean().item()

print(f"\n📊 수치 비교:")
print(f"   - Max Difference: {max_diff:.8f}")
print(f"   - Mean Difference: {mean_diff:.8f}")

# 허용 오차
tolerance = 1e-3 if orig_model.dtype == torch.float16 else 1e-5

if max_diff < tolerance:
    print("\n🎉 성공! 두 모델의 결과가 일치합니다 (허용 오차 내).")
else:
    print("\n❌ 실패! 두 모델의 결과가 너무 다릅니다.")

   - 원본 모델 Forward...
   - Standard 모델 Forward...

💬 모델 출력 비교 (Next Token):
   - 원본 모델 예측: ' I'
   - Standard 예측 : ' I'

📊 수치 비교:
   - Max Difference: 0.15625000
   - Mean Difference: 0.02026367

❌ 실패! 두 모델의 결과가 너무 다릅니다.


In [6]:
# 문장 생성 테스트 (Generation)
print("\n📝 문장 생성 테스트:")
gen_inputs = tokenizer(input_text, return_tensors="pt")

# 원본 모델 생성
print("   - 원본 모델 생성 중...")
orig_gen = orig_model.generate(**gen_inputs.to(device_orig), max_new_tokens=100)
print(f"   [Original]: {tokenizer.decode(orig_gen[0], skip_special_tokens=True)}")

# Standard 모델 생성
print("   - Standard 모델 생성 중...")
std_gen = std_model.generate(**gen_inputs.to(device_std), max_new_tokens=100)
print(f"   [Standard]: {tokenizer.decode(std_gen[0], skip_special_tokens=True)}")


📝 문장 생성 테스트:
   - 원본 모델 생성 중...
   [Original]: Hello, I want to verify the model structure. ... I got the last layer as 5 as 3 in input. So what would be the last layer? I am also using Tensorflow 2.9.1. For the last layer i used something like ... The model structure is as follows and I had already tried ... I want to reduce the dimensionality to one. I tried different ways. I also came up with a solution that I used to check if the model structure is correct and it is indeed correct. ... The error that
   - Standard 모델 생성 중...
   [Standard]: Hello, I want to verify the model structure. Using the command `net = Net(10, 10)` or `net = nn.Sequential(...)` I only get a model with `numel = 0` (no parameters). But when I run it with `x = torch.rand([16,3,32,32]) ; net(x)`, the network seems to produce outputs that are *not* zeros, and this seems to be *right* as if it in fact contains weights. Why?` So they may see num


In [ ]:
import torch

std_model.eval()
orig_model.eval()
std_model.train()

# 2. Hook 함수 정의 (중간값 캡처용)
# 메모리 절약을 위해 캡처 즉시 CPU로 내립니다 (GPU간 직접 통신 에러 방지)
def get_hook(name, storage):
    def hook(model, input, output):
        if isinstance(output, tuple):
            output = output[0]
        # 즉시 CPU로 이동하여 저장 (나중에 device_comp로 옮겨서 비교)
        storage[name] = output.detach().cpu().float()
    return hook

orig_storage = {}
std_storage = {}

# 3. Hook 등록 (Layer 0번만 집중 분석)
# 모델 구조에 따라 이름이 다를 수 있어 안전하게 접근합니다.
layer_idx = 0

# (1) Original Model Hooks
orig_layer = orig_model.model.layers[layer_idx]
orig_layer.self_attn.register_forward_hook(get_hook("L0_Attn", orig_storage))
orig_layer.mlp.register_forward_hook(get_hook("L0_MLP", orig_storage))
orig_layer.register_forward_hook(get_hook("L0_Output", orig_storage))

# (2) Standard Model Hooks
std_layer = std_model.model.layers[layer_idx]
std_layer.self_attn.register_forward_hook(get_hook("L0_Attn", std_storage))

# Standard 모델의 MLP 모듈 이름 찾기 (mlp 또는 experts일 수 있음)
if hasattr(std_layer, 'experts'):
    std_layer.experts.register_forward_hook(get_hook("L0_MLP", std_storage))
elif hasattr(std_layer, 'mlp'):
    std_layer.mlp.register_forward_hook(get_hook("L0_MLP", std_storage))
elif hasattr(std_layer, 'block_sparse_moe'): # Mixtral 이름
    std_layer.block_sparse_moe.register_forward_hook(get_hook("L0_MLP", std_storage))
else:
    print("⚠️ Standard 모델에서 MLP/Experts 모듈을 찾을 수 없습니다.")

std_layer.register_forward_hook(get_hook("L0_Output", std_storage))
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_directory)

# 4. Forward 실행
print("\n🧪 디버깅용 Forward 시작...")
input_text = "Hello, I want to verify the model structure."
inputs = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    # 원본 모델 추론
    print(f"   - 원본 모델 Forward on {device_orig}...")
    _ = orig_model(**inputs.to(device_orig))
    
    # Standard 모델 추론
    print(f"   - Standard 모델 Forward on {device_std}...")
    _ = std_model(**inputs.to(device_std))

# 5. 정밀 비교 분석
print("\n🕵️‍♀️ 레이어별 오차 분석 시작...")

targets = ["L0_Attn", "L0_MLP", "L0_Output"]
metrics = {}

for key in targets:
    if key not in orig_storage or key not in std_storage:
        print(f"⚠️ {key} 데이터가 누락되었습니다.")
        continue

    # 비교를 위해 device_comp로 이동
    val_orig = orig_storage[key].to(device_comp)
    val_std = std_storage[key].to(device_comp)
    
    # 오차 계산
    diff = (val_orig - val_std).abs()
    max_diff = diff.max().item()
    mean_diff = diff.mean().item()
    
    print(f"\n📍 [{key}] 비교:")
    print(f"   - Max Diff : {max_diff:.8f}")
    print(f"   - Mean Diff: {mean_diff:.8f}")
    
    if max_diff > 1e-3: # fp16 기준 허용치 초과 시
        print(f"   🚨 [FAIL] 오차 발생! 문제가 시작된 곳일 수 있습니다.")
    else:
        print(f"   ✅ [PASS] 정상 범위")

print("\n" + "="*30)
print("💡 분석 가이드:")
print("1. L0_Attn에서 FAIL이 떴다면? -> Attention 모듈 구현 문제 (Sink 등)")
print("2. L0_Attn은 통과했는데 L0_MLP에서 FAIL? -> Experts/Router 구현 문제 (TopK 순서, Slicing 등)")
print("3. 둘 다 통과했는데 L0_Output FAIL? -> Residual Connection이나 Norm 문제")

In [ ]:
import torch

print("⚖️ [최종 판결] Expert 0번 연산 정밀 검증")

# 1. 검증용 랜덤 입력 데이터 생성 (Batch=1, Hidden Size)
# 모델과 동일한 dtype/device 사용
device = next(std_model.parameters()).device
dtype = next(std_model.parameters()).dtype
dummy_input = torch.randn(1, 2880, device=device, dtype=dtype) # Hidden Size 2880 가정 (확인 필요)

# -------------------------------------------------------------------------
# A. Original Model의 Expert 0번 수동 계산
# -------------------------------------------------------------------------
# 가중치 직접 추출
# GptOss 원본 가중치 Shape: (Hidden, 2*Inter) 또는 (In, Out) 가정
w_orig = orig_model.model.layers[0].mlp.experts.gate_up_proj[0]
b_orig = orig_model.model.layers[0].mlp.experts.gate_up_proj_bias[0]
w_down_orig = orig_model.model.layers[0].mlp.experts.down_proj[0]
b_down_orig = orig_model.model.layers[0].mlp.experts.down_proj_bias[0]

# 상수 확인
alpha = getattr(orig_model.model.layers[0].mlp.experts, 'alpha', 1.702)
limit = getattr(orig_model.model.layers[0].mlp.experts, 'limit', 7.0)

print(f"   Note: Alpha={alpha}, Limit={limit}")

# [수동 Forward]
# 1. Gate Up
# 원본 가중치 Shape에 따라 Transpose 필요 여부 결정 (보통 원본은 In, Out 저장)
gate_up_orig = dummy_input.to(w_orig.device) @ w_orig + b_orig 

# 2. Slice & Clamp
gate_orig = gate_up_orig[..., ::2].clamp(max=limit)
up_orig = gate_up_orig[..., 1::2].clamp(min=-limit, max=limit)

# 3. Activation
glu_orig = gate_orig * torch.sigmoid(gate_orig * alpha)

# 4. Final (+1 포함 확인)
out_orig = (up_orig + 1) * glu_orig

# 5. Down Proj
final_orig = out_orig @ w_down_orig + b_down_orig


# -------------------------------------------------------------------------
# B. Standard Model의 Expert 0번 모듈 실행
# -------------------------------------------------------------------------
expert_0 = std_model.model.layers[0].mlp.experts.experts[0]

# [모듈 Forward]
with torch.no_grad():
    final_std = expert_0(dummy_input)

# -------------------------------------------------------------------------
# C. 결과 비교
# -------------------------------------------------------------------------
diff = (final_orig.to(final_std.device) - final_std).abs().max().item()

print(f"\n📊 결과 비교 (입력값 통일됨):")
print(f"   - Original Output (First 5): {final_orig[0, :100].tolist()}")
print(f"   - Standard Output (First 5): {final_std[0, :100].tolist()}")
print(f"   - Max Difference: {diff:.8f}")

if diff < 1e-3:
    print("\n🎉 [축하합니다] Expert 연산은 완벽하게 일치합니다!")
    print("   이전에 발생한 오차는 '서로 다른 토큰'을 비교해서 생긴 해프닝입니다.")
else:
    print("\n🚨 [여전히 불일치] 입력이 같은데도 값이 다릅니다.")
    print("   -> 가중치 Transpose 방향이 틀렸거나, Alpha/Limit 값이 다릅니다.")

In [ ]:
diff = (final_orig.to(final_std.device) - final_std).abs().max()


In [ ]:
import torch

print("⚖️ [최종 판결] 수식 변형 테스트 (3가지 Case)")

# 1. 데이터 준비 (Standard 모델 가중치 사용)
device = next(std_model.parameters()).device
dtype = next(std_model.parameters()).dtype

# 정밀 비교를 위해 Float32로 변환하여 테스트 (오차 원인인 정밀도 문제 제거)
dummy_input = torch.randn(1, 2880, device=device).float() 

# Standard Expert 0번 가중치 가져오기 (Float32 변환)
expert_module = std_model.model.layers[0].mlp.experts.experts[0]
w_gate_up = expert_module.gate_up_proj.weight.t().float() # (Hidden, 2*Inter)
b_gate_up = expert_module.gate_up_proj.bias.float()
w_down = expert_module.down_proj.weight.t().float()
b_down = expert_module.down_proj.bias.float()

alpha = 1.702
limit = 7.0

# 2. 공통 연산 (Linear -> Slice -> Clamp -> Sigmoid)
gate_up = dummy_input @ w_gate_up + b_gate_up

# [정석 Slice]
gate = gate_up[..., ::2].clamp(max=limit)
up = gate_up[..., 1::2].clamp(min=-limit, max=limit)

# [반대 Slice] (혹시 순서가 반대일 경우)
gate_swap = gate_up[..., 1::2].clamp(max=limit)
up_swap = gate_up[..., ::2].clamp(min=-limit, max=limit)

glu = gate * torch.sigmoid(gate * alpha)
glu_swap = gate_swap * torch.sigmoid(gate_swap * alpha)


# 3. 세 가지 Case 계산
# Case A: (up + 1) * glu  (현재 유력)
out_A = (up + 1) * glu
res_A = out_A @ w_down + b_down

# Case B: up * glu  (+1 제거)
out_B = up * glu
res_B = out_B @ w_down + b_down

# Case C: Slice 반대 + (up + 1)
out_C = (up_swap + 1) * glu_swap
res_C = out_C @ w_down + b_down


# 4. Standard 모델 실제 출력 (비교 대상)
# 입력도 float32로 넣어서 정밀도 문제 배제
with torch.no_grad():
    # 모델 내부가 BF16이라도 입력에 맞춰질 수 있으니 강제 형변환 필요할 수 있음
    # 여기서는 모델이 BF16이면 어쩔 수 없이 오차 발생.
    # 그래도 경향성은 보임.
    target_out = expert_module(dummy_input.to(dtype)).float() 


# 5. 오차 비교
diff_A = (res_A - target_out).abs().max().item()
diff_B = (res_B - target_out).abs().max().item()
diff_C = (res_C - target_out).abs().max().item()

print(f"\n📊 오차 비교 결과 (vs Standard 실제 출력):")
print(f"   1️⃣ Case A [(up+1) * glu]: {diff_A:.8f}")
print(f"   2️⃣ Case B [up * glu]    : {diff_B:.8f}")
print(f"   3️⃣ Case C [Swap Slice]  : {diff_C:.8f}")

print("-" * 50)
if diff_A < diff_B and diff_A < diff_C:
    print("✅ Case A (현재 코드)가 가장 정확합니다.")
    print("   -> 0.35 오차는 BFloat16의 정밀도 차이일 뿐입니다. (무시 가능)")
elif diff_B < diff_A:
    print("🚨 Case B (+1 제거)가 더 정확합니다!")
    print("   -> 코드에서 +1을 지우세요.")
elif diff_C < diff_A:
    print("🚨 Case C (Slice 반대)가 더 정확합니다!")
    print("   -> Slicing 순서를 바꾸세요.")

In [ ]:
import torch

# ==========================================
# 1. 디버깅용 Hook 함수 정의
# ==========================================
orig_dump = {}
std_dump = {}

def hook_router(storage, name):
    def hook(model, input, output):
        # Router 출력: (scores, indices)
        storage[f"{name}_scores"] = output[0].detach().cpu().float()
        storage[f"{name}_indices"] = output[1].detach().cpu()
    return hook

def hook_experts(storage, name):
    def hook(model, input, output):
        # Experts 출력: Tensor
        storage[f"{name}_out"] = output.detach().cpu().float()
    return hook

def hook_mlp_input(storage, name):
    def hook(model, input, output):
        # MLP 입력: (hidden_states,) 튜플
        storage[f"{name}_in"] = input[0].detach().cpu().float()
    return hook

# ==========================================
# 2. Hook 등록 (Layer 0 집중 분석)
# ==========================================
# 모델 구조에 따라 접근 경로가 다를 수 있으니 확인 필요
# 보통: model.layers[0].mlp

# [Original Model Hook]
layer_orig = orig_model.model.layers[0].mlp
layer_orig.register_forward_hook(hook_mlp_input(orig_dump, "MLP"))
layer_orig.router.register_forward_hook(hook_router(orig_dump, "Router"))
layer_orig.experts.register_forward_hook(hook_experts(orig_dump, "Experts"))

# [Standard Model Hook]
layer_std = std_model.model.layers[0].mlp
layer_std.register_forward_hook(hook_mlp_input(std_dump, "MLP"))
layer_std.router.register_forward_hook(hook_router(std_dump, "Router"))
layer_std.experts.register_forward_hook(hook_experts(std_dump, "Experts"))

print("🪝 Hook 설치 완료. 내부 값을 캡처할 준비가 되었습니다.")

# ==========================================
# 3. Forward 실행
# ==========================================
device_orig = next(orig_model.parameters()).device
device_std = next(std_model.parameters()).device

# 더미 입력
input_text = "Hello, verifying MLP internals."
inputs = tokenizer(input_text, return_tensors="pt")

print("\n🚀 Forward 실행 중...")
with torch.no_grad():
    _ = orig_model(**inputs.to(device_orig))
    _ = std_model(**inputs.to(device_std))

# ==========================================
# 4. 단계별 정밀 비교
# ==========================================
print("\n🕵️‍♀️ [MLP 내부 해부 결과]")

# (1) MLP 입력값 비교 (여기서 다르면 이전 레이어 문제)
diff_in = (orig_dump["MLP_in"] - std_dump["MLP_in"]).abs()
print(f"\n1️⃣ MLP 입력값 (Input Hidden States)")
print(f"   - Max Diff: {diff_in.max().item():.8f}")
if diff_in.max().item() > 1e-4:
    print("   🚨 [WARNING] 입력부터 이미 다릅니다! 이전 레이어(Attention/Norm)를 보세요.")
else:
    print("   ✅ 입력값 정상 (동일함)")

# (2) Router Indices 비교 (정수형, 무조건 똑같아야 함)
indices_orig = orig_dump["Router_indices"].long()
indices_std = std_dump["Router_indices"].long()
# 같은지 비교 (True/False)
match = (indices_orig == indices_std).all().item()

print(f"\n2️⃣ Router Indices (선택된 전문가 번호)")
print(f"   - 원본(일부): {indices_orig[0, :5].tolist()} ...")
print(f"   - 표준(일부): {indices_std[0, :5].tolist()} ...")
if match:
    print("   ✅ Indices 완전 일치 (라우팅 로직 정상)")
else:
    print("   ❌ [FAIL] 서로 다른 전문가를 선택했습니다!")
    print("      -> Router의 가중치(gate.weight)가 잘못 이식되었거나, 입력값이 달라서 발생했습니다.")

# (3) Router Scores 비교 (확률값)
diff_scores = (orig_dump["Router_scores"] - std_dump["Router_scores"]).abs()
print(f"\n3️⃣ Router Scores (Top-K 확률값)")
print(f"   - Max Diff: {diff_scores.max().item():.8f}")
if diff_scores.max().item() > 1e-4:
    print("   ❌ [FAIL] 확률값이 다릅니다. Router의 Linear 연산이나 Softmax 순서를 확인하세요.")
else:
    print("   ✅ Scores 정상 (가중치 연산 정상)")

# (4) Experts Output 비교 (최종 결과)
diff_out = (orig_dump["Experts_out"] - std_dump["Experts_out"]).abs()
print(f"\n4️⃣ Experts Output (최종 결과)")
print(f"   - Max Diff: {diff_out.max().item():.8f}")

if diff_out.max().item() > 1e-3:
    if match and diff_scores.max().item() < 1e-4:
        print("   🚨 [결론] 입력/라우팅/확률은 다 맞는데 결과만 틀립니다.")
        print("      -> 범인은 'GptOssLayer' 내부 로직입니다.")
        print("      -> 체크포인트: 1) 가중치 Transpose 여부, 2) Activation(+1), 3) Slicing 순서")
    else:
        print("   🚨 [결론] 앞 단계(Indices/Scores)가 틀려서 결과도 틀린 것입니다.")
else:
    print("   🎉 [SUCCESS] MLP 내부 동작이 완벽하게 일치합니다.")

# # 메모리 정리
# orig_dump.clear()
# std_dump.clear()

In [ ]:
# 🔍 가중치 직접 비교 스크립트

# 1. Layer 0의 첫 번째 Expert 선택
orig_expert_bias = orig_model.model.layers[0].mlp.experts.down_proj_bias[0] # (Hidden,)
std_expert_bias = std_model.model.layers[0].mlp.experts.experts[0].down_proj.bias   # (Hidden,)

print(f"🕵️ Down Proj Bias 비교 (Expert 0)")
print(f"   - Original (First 5): {orig_expert_bias[:5].tolist()}")
print(f"   - Standard (First 5): {std_expert_bias[:5].tolist()}")

# 오차 계산
bias_diff = (orig_expert_bias - std_expert_bias.to(orig_expert_bias.device)).abs().max().item()
print(f"   - Max Bias Diff: {bias_diff:.8f}")

if bias_diff > 1e-4:
    print("\n🚨 [범인 검거] Bias 가중치가 서로 다릅니다!")
    print("   -> transfer_weights 함수에서 'bias' 복사 로직을 점검하세요.")
else:
    print("\n✅ Bias는 일치합니다. (미궁 속으로...)")

# 2. Gate Up Bias도 확인
orig_gate_bias = orig_model.model.layers[0].mlp.experts.gate_up_proj_bias[0]
std_gate_bias = std_model.model.layers[0].mlp.experts.experts[0].gate_up_proj.bias

print(f"\n🕵️ Gate-Up Proj Bias 비교 (Expert 0)")
bias_diff_gate = (orig_gate_bias - std_gate_bias.to(orig_gate_bias.device)).abs().max().item()
print(f"   - Max Bias Diff: {bias_diff_gate:.8f}")

In [ ]:
# 수정된 수식 검증 코드

# 1. 데이터 및 가중치 준비
device = device_orig # 원본 모델이 있는 GPU
mlp_input = orig_dump["MLP_in"].to(device) # (batch, seq, hidden), float32

# 가중치를 가져올 때 .float()로 형변환을 미리 해줍니다.
# Expert 0번 기준
w_gate_up = orig_model.model.layers[0].mlp.experts.gate_up_proj[0].float().to(device)
b_gate_up = orig_model.model.layers[0].mlp.experts.gate_up_proj_bias[0].float().to(device)
w_down = orig_model.model.layers[0].mlp.experts.down_proj[0].float().to(device)
b_down = orig_model.model.layers[0].mlp.experts.down_proj_bias[0].float().to(device)

# GptOss 파라미터
alpha = 1.702
limit = 7.0

def manual_forward(x, formula_type="standard"):
    # (1) Linear: Gate_Up
    # GptOss 원본 가중치 shape은 보통 (Hidden, 2*Inter)이므로 x @ w 그대로 가능
    # 만약 Shape 에러가 나면 w_gate_up.t()를 시도해야 함
    gate_up = x @ w_gate_up + b_gate_up
    
    # (2) Slicing
    if formula_type == "swap_slice":
        # 반대로 슬라이싱 테스트
        gate, up = gate_up[..., 1::2], gate_up[..., ::2]
    else:
        # 정석 슬라이싱 (Gate=짝수, Up=홀수)
        gate, up = gate_up[..., ::2], gate_up[..., 1::2]
        
    # (3) Activation & Gating
    gate = gate.clamp(max=limit)
    up = up.clamp(min=-limit, max=limit)
    
    glu = gate * torch.sigmoid(gate * alpha)
    
    # (4) ★★★ 여기가 핵심 검증 포인트 ★★★
    if formula_type == "no_plus_one":
        # Case B: +1 제거 (Standard SwiGLU)
        gated_output = up * glu 
    elif formula_type == "plus_one_swapped":
        # Case X: (glu + 1) * up 같은 변종 확인용 (가능성 낮음)
        gated_output = (glu + 1) * up
    else:
        # Case A: +1 포함 (GptOss Current Code)
        gated_output = (up + 1) * glu
        
    # (5) Down Proj
    # GptOss 원본 DownProj는 (Inter, Hidden)이므로 x @ w 가능
    out = gated_output @ w_down + b_down
    return out

print("\n🕵️ 수식 범인 찾기 재시작 (dtype 수정됨)...")

with torch.no_grad():
    # 1. 현재 로직 (up + 1)
    try:
        calc_A = manual_forward(mlp_input, "standard")
        print(f"   [Case A: (up+1)] Mean: {calc_A.mean():.4f}, Max: {calc_A.max():.4f}")
    except Exception as e:
        print(f"   [Case A] 실패: {e}")

    # 2. +1 제거 (유력 용의자)
    try:
        calc_B = manual_forward(mlp_input, "no_plus_one")
        print(f"   [Case B: up only] Mean: {calc_B.mean():.4f}, Max: {calc_B.max():.4f}")
    except Exception as e:
         print(f"   [Case B] 실패: {e}")

    # 3. 슬라이싱 반대
    try:
        calc_C = manual_forward(mlp_input, "swap_slice")
        print(f"   [Case C: Swapped] Mean: {calc_C.mean():.4f}, Max: {calc_C.max():.4f}")
    except Exception as e:
         print(f"   [Case C] 실패: {e}")

# 비교를 위한 Standard 모델의 실제 출력값 (Experts_out) 정보
std_out_mean = std_dump["Experts_out"].float().mean().item()
std_out_max = std_dump["Experts_out"].float().max().item()

print(f"\n📏 Standard 모델(현재 코드)의 실제 출력:")
print(f"   - Mean: {std_out_mean:.4f}")
print(f"   - Max : {std_out_max:.4f}")
print("\n👉 위 Case 중 Standard 모델 값과 가장 '다른' 것이 정답(원본)일 수 있습니다.")
# 하지만 가장 좋은 건 'orig_model'의 출력값과 비슷한 걸 찾는 것입니다. 
# 다만 orig_model 출력은 Router가 적용된 합산값이라 직접 비교가 힘드니,
# 이 수동 계산 값이 '튀지 않고 정상적인 분포'를 보이는지 확인하세요.

In [ ]:
input_text = "Hello, I want to verify the model structure."
inputs = tokenizer(input_text, return_tensors="pt").to(orig_model.device)
deivce_std = 'cuda:0'
device_orig = 'cuda:1'
device_comp = 'cuda:2'
# 디버깅용: MLP 직전 값(입력) 비교
def get_inputs_hook(name, storage):
    def hook(model, input, output):
        # input은 튜플이므로 첫번째 요소만 가져옴
        storage[name] = input[0].detach().cpu().float()
    return hook

orig_in = {}
std_in = {}

# Layer 0의 MLP "입력값"을 캡처 (Attention + Residual + Norm을 거친 직후)
orig_model.model.layers[0].mlp.register_forward_hook(get_inputs_hook("MLP_Input", orig_in))
# Standard 모델은 experts 혹은 mlp 등 이름에 맞춰 수정
std_model.model.layers[0].mlp.experts.register_forward_hook(get_inputs_hook("MLP_Input", std_in))

# Forward 실행
with torch.no_grad():
    orig_model(**inputs.to(device_orig))
    std_model(**inputs.to(device_std))

# 비교
diff = (orig_in["MLP_Input"].to(device_comp) - std_in["MLP_Input"].to(device_comp)).abs()
print(f"MLP 입력값 오차: {diff.max().item()}")